# 01 Preprocessing — Stage 1 Dataset Build

This notebook runs the complete Stage 1 preprocessing pipeline and exports processed files.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.load_data import load_stage1_data
from src.preprocessing.clean import handle_missing
from src.preprocessing.target_builder import build_targets
from src.preprocessing.engineer_features import engineer_features
from src.preprocessing.encode import encode_features

print('Project root:', PROJECT_ROOT)

Project root: b:\PBCS\Major Project\BarrierLens_MP_G25_P48


In [ ]:
df = load_stage1_data()
print('Loaded shape:', df.shape)
df.head(2)

Raw dataset shape: (706, 109)
After column selection: (706, 24)
Loaded shape: (706, 24)


,District Names,State/UT,Population living in households with electricity (%),Population living in households with an improved drinking-water source1 (%),Population living in households that use an improved sanitation facility2 (%),Households using clean fuel for cooking3 (%),Households using iodized salt (%),Female population age 6 years and above who ever attended school (%),Women (age 15-49) who are literate4 (%),Women (age 15-49) with 10 or more years of schooling (%),...,Births in the 5 years preceding the survey that are third or higher order (%),Total Unmet need for Family Planning (Currently Married Women Age 15-49 years)7 (%),Households with any usual member covered under a health insurance/financing scheme (%),All women age 15-49 years who are anaemic22 (%),Women (age 15-49 years) whose Body Mass Index (BMI) is below normal (BMI <18.5 kg/m2)21 (%),Institutional births (in the 5 years before the survey) (%),Mothers who had at least 4 antenatal care visits (for last birth in the 5 years before the survey) (%),Births attended by skilled health personnel (in the 5 years before the survey)10 (%),Children age 12-23 months fully vaccinated based on information from either vaccination card or mother's recall11 (%),Mothers who received postnatal care from a doctor/nurse/LHV/ANM/midwife/other health personnel within 2 days of delivery (for last birth in the 5 years before the survey) (%)
0,Nicobars,Andaman & Nicobar Islands,97.9,98.8,83.5,56.9,99.4,78.0,87.5,53.5,...,0.0,9.5,2.7,38.3,8.2,97.8,71.7,98.6,(64.2),85.1
1,North & Middle Andaman,Andaman & Nicobar Islands,93.2,92.2,86.4,61.3,99.9,82.7,84.0,41.0,...,1.5,5.8,2.1,62.1,8.6,97.7,79.2,98.3,*,92.5


In [ ]:
df = handle_missing(df)
df = build_targets(df)
df = engineer_features(df)
df = encode_features(df)

print('Post-encoding shape:', df.shape)
print(df[['target_household', 'target_logistic', 'target_facility']].mean())

Total missing values before imputation: 252
Total missing values after imputation: 0
Post-encoding shape: (706, 57)
target_household    0.5
target_logistic     0.5
target_facility     0.5
dtype: float64


In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

X_full = df[[c for c in df.columns if not c.startswith('target_')]].copy()
X_full.to_csv(processed_dir / 'X_features.csv', index=False)
df['target_household'].to_csv(processed_dir / 'y_household.csv', index=False)
df['target_logistic'].to_csv(processed_dir / 'y_logistic.csv', index=False)
df['target_facility'].to_csv(processed_dir / 'y_facility.csv', index=False)

print('Saved processed files to:', processed_dir)
print('X_features shape:', X_full.shape)
for t in ['target_household', 'target_logistic', 'target_facility']:
    print(t, df[t].value_counts().to_dict())

Saved processed files to: b:\PBCS\Major Project\BarrierLens_MP_G25_P48\data\processed
X_features shape: (706, 54)
target_household {1: 353, 0: 353}
target_logistic {0: 353, 1: 353}
target_facility {0: 353, 1: 353}
